# Geographic performance — Supplementary exploration

Author-origin quantitative analysis retained. See the English consolidated case study for validated interpretation and executive storytelling. The original explorations were produced before the portfolio restructuring and may contain Italian in analytical print messages.

## Setup — caricamento e controlli

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 110, 'axes.titleweight':'semibold', 'axes.labelsize':10, 'font.size':10})
pd.set_option('display.max_colwidth', 75)
pd.set_option('display.max_rows', 80)

candidates = [
    Path.cwd() / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent.parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd() / 'adidas_sales_features.csv',
    Path.cwd().parent / 'adidas_sales_features.csv',
]
file_input = next((p for p in candidates if p.is_file()), None)
if file_input is None:
    raise FileNotFoundError('Manca output/adidas_sales_features.csv. Esegui i notebook 01 e 02 del progetto, poi avvia questo notebook dalla root o dalla cartella notebooks/.')
df = pd.read_csv(file_input, parse_dates=['invoice_date'], dtype={'retailer_id': 'string'})
expected = {'retailer','invoice_date','region','state','city','product','price_per_unit',
            'units_sold','total_sales','operating_profit','operating_margin','sales_method',
            'year','month_number','month_name_it','year_month','weekday_number',
            'weekday_name_it','is_weekend','season_it','product_gender','product_category'}
assert expected.issubset(df.columns), f'Colonne mancanti: {expected-set(df.columns)}'
assert not df['invoice_date'].isna().any(), 'Date mancanti'
assert np.allclose(df['total_sales'], df['price_per_unit']*df['units_sold'],atol=.011), 'Ricavi non coerenti'
assert np.allclose(df['operating_profit'],df['total_sales']*df['operating_margin'],atol=.011), 'Profitto non coerente'

EURO_MONTH=['Gennaio','Febbraio','Marzo','Aprile','Maggio','Giugno','Luglio','Agosto','Settembre','Ottobre','Novembre','Dicembre']
DAYS=['Lunedì','Martedì','Mercoledì','Giovedì','Venerdì','Sabato','Domenica']
SEASONS=['Inverno','Primavera','Estate','Autunno']

def units(x): return f'{x:,.0f}'.replace(',', '.')
def usd(x): return f'${x:,.2f}' if abs(x)<1000 else f'${x:,.0f}'
def percent(x): return 'N/D' if pd.isna(x) else f'{x:,.2f}%'
def safe(a,b): return np.nan if b==0 or pd.isna(b) else a/b
def avg(x): return f'{x:,.2f}'
def note(message): display(Markdown('**Risultato:** '+str(message)))
def caution(message): display(Markdown('**Limite interpretativo:** '+str(message)))
def data_table(frame, rows=80):
    display(frame.head(rows).reset_index(drop=True).round(2))
    if len(frame)>rows:
        display(Markdown(f'_Visualizzate {rows} righe su {len(frame)}. La tabella completa rimane disponibile nella variabile Python._'))

def summary(dims, data=None):
    frame=df if data is None else data
    group=frame.groupby(dims,dropna=False,observed=True)
    result=group.agg(units=('units_sold','sum'),revenue=('total_sales','sum'),
                     profit=('operating_profit','sum'),rows=('units_sold','size'),
                     price_mean=('price_per_unit','mean'),days=('invoice_date','nunique')).reset_index()
    result['margin_pct']=np.where(result.revenue!=0,100*result.profit/result.revenue,np.nan)
    result['realized_price']=np.where(result.units!=0,result.revenue/result.units,np.nan)
    result['profit_unit']=np.where(result.units!=0,result.profit/result.units,np.nan)
    return result

def dominance(frame,dims,category,measure='units'):
    x=frame.sort_values(dims+[measure,category],ascending=[True]*len(dims)+[False,True])
    first=x.drop_duplicates(subset=dims,keep='first').copy()
    top=frame.groupby(dims)[measure].transform('max')
    tied=frame[frame[measure].eq(top)].groupby(dims,dropna=False)[category].nunique().reset_index(name='ex_aequo')
    return first.merge(tied,on=dims,how='left')

def bar(frame,label,value,title,top=12,ascending=False,ax=None):
    d=frame.sort_values(value,ascending=ascending).head(top).copy()
    if ax is None: fig,ax=plt.subplots(figsize=(11,max(4,0.40*len(d)+1.5)))
    else: fig=ax.figure
    d=d.iloc[::-1]
    ax.barh(d[label].astype(str),d[value]);ax.set_title(title);ax.set_xlabel(value.replace('_',' ').title());ax.grid(axis='x',alpha=.25)
    ax.tick_params(axis='y',labelsize=9)
    fig.tight_layout()
    return ax

def twobars(frame,label,metrics,titles,top=12):
    fig,axes=plt.subplots(1,2,figsize=(15,max(4,0.43*min(top,len(frame))+1)))
    for ax,m,t in zip(axes,metrics,titles):bar(frame,label,m,t,top=top,ax=ax)
    plt.show()

def stack(data,index,columns,measure,title,top=14,shares=True):
    g=data.pivot_table(index=index,columns=columns,values=measure,aggfunc='sum',fill_value=0)
    if g.empty:
        note('Non sono presenti dati per questo confronto.');return
    g=g.loc[g.sum(axis=1).sort_values(ascending=False).head(top).index]
    if shares: g=g.div(g.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    ax=g.plot(kind='bar',stacked=True,figsize=(max(10, min(16,len(g)*0.7+5)),5.5),width=.80)
    ax.set_title(title);ax.set_ylabel('Quota sul volume (%)' if shares else measure)
    ax.set_xlabel('');ax.legend(title=columns,bbox_to_anchor=(1.01,1),loc='upper left')
    plt.xticks(rotation=40,ha='right');plt.tight_layout();plt.show()

def heat(data,index,columns,value,title,top=25,normalize=False):
    piv=data.pivot_table(index=index,columns=columns,values=value,aggfunc='sum',fill_value=0)
    if piv.empty: return
    piv=piv.loc[piv.sum(axis=1).sort_values(ascending=False).head(top).index]
    if normalize: piv=piv.div(piv.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    fig,ax=plt.subplots(figsize=(max(8,1.15*len(piv.columns)+4),max(4,.36*len(piv)+1.8)))
    im=ax.imshow(piv.to_numpy(),aspect='auto',cmap='Blues')
    ax.set_xticks(np.arange(len(piv.columns)),piv.columns.astype(str),rotation=35,ha='right')
    ax.set_yticks(np.arange(len(piv.index)),piv.index.astype(str));ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.80,label='Quota (%)' if normalize else value)
    fig.tight_layout();plt.show()

def lineplot(frame,x,y,title,color_key=None):
    fig,ax=plt.subplots(figsize=(12,5))
    if color_key is None:
        t=frame.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3)
    else:
        for name,piece in frame.groupby(color_key,observed=True):
            t=piece.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3,label=str(name))
        ax.legend(title=color_key)
    ax.set_title(title);ax.set_ylabel(y);ax.tick_params(axis='x',rotation=70,labelsize=8)
    ax.grid(alpha=.25);fig.tight_layout();plt.show()

note(f'Caricate {units(len(df))} righe • periodo {df.invoice_date.min():%Y-%m-%d} — {df.invoice_date.max():%Y-%m-%d} • nessun CSV verrà esportato.')

## Which regions, states and cities lead in volume?

In [ ]:
rg=summary(['region']);st=summary(['region','state']);ct=summary(['region','state','city']);ct['place']=ct.state+' / '+ct.city
ar=rg.loc[rg.units.idxmax()];ass=st.loc[st.units.idxmax()];ac=ct.loc[ct.units.idxmax()]
note(f"Regione leader: {ar.region} ({units(ar.units)} unità). Stato: {ass.state} ({units(ass.units)}). Città: {ac.city}, {ac.state} ({units(ac.units)}).")
fig,axes=plt.subplots(1,3,figsize=(20,7));bar(rg,'region','units','Regioni',ax=axes[0]);bar(st,'state','units','Top 10 stati',top=10,ax=axes[1]);bar(ct,'place','units','Top 10 città',top=10,ax=axes[2]);plt.show()
data_table(rg);data_table(st.sort_values('units',ascending=False));data_table(ct.sort_values('units',ascending=False))

## Which products dominate different locations?

In [ ]:
p=summary(['region','state','city','product']);c=summary(['region','state','city','product_category'])
w_region=dominance(summary(['region','product']),['region'],'product')
w_state=dominance(summary(['region','state','product']),['region','state'],'product')
w_city=dominance(p,['region','state','city'],'product')
w_cat=dominance(c,['region','state','city'],'product_category')
note('I prodotti dominanti sono definiti per unità, non per profitto. I ranking per stato e città includono l’intera gerarchia geografica.')
heat(summary(['region','product']),'region','product','units','Volumi: regione × prodotto')
data_table(w_region[['region','product','units','ex_aequo']]);data_table(w_state[['region','state','product','units','ex_aequo']]);data_table(w_city[['region','state','city','product','units','ex_aequo']]);data_table(w_cat[['region','state','city','product_category','units','ex_aequo']])

## Where do individual retailers concentrate sales?

In [ ]:
rg=summary(['retailer','region']);st=summary(['retailer','state']);ct=summary(['retailer','state','city'])
w_rg=dominance(rg,['retailer'],'region');w_st=dominance(st,['retailer'],'state');w_ct=dominance(ct,['retailer'],'city')
note('Territori dominanti per retailer determinati in base al numero di unità, con tabella che distingue stato e città.')
heat(rg,'retailer','region','units','Retailer × regione: unità')
data_table(w_rg[['retailer','region','units']]);data_table(w_st[['retailer','state','units']]);data_table(w_ct[['retailer','state','city','units']]);data_table(rg)

## Where are margin and profit per unit strongest?

In [ ]:
t=summary(['region','state']);r=summary(['region']);best=t.loc[t.margin_pct.idxmax()]
note(f"Margine aggregato massimo fra gli stati rappresentati: {best.state} ({percent(best.margin_pct)}, {units(best.units)} unità). L’affidabilità descrittiva va letta insieme ai volumi.")
fig,axes=plt.subplots(1,2,figsize=(15,6));bar(t,'state','margin_pct','Top 12 margini per stato',top=12,ax=axes[0]);bar(t,'state','profit_unit','Top 12 profitti per unità',top=12,ax=axes[1]);plt.show()
data_table(r.sort_values('margin_pct',ascending=False));data_table(t.sort_values('margin_pct',ascending=False))

## How does partner/category/channel composition differ by region?

In [ ]:
frames=[('retailer',summary(['region','retailer'])),('product_category',summary(['region','product_category'])),('sales_method',summary(['region','sales_method']))]
note('Le percentuali qui sono quote di unità ENTRO ciascuna regione; non quote nazionali o di mercato.')
fig,axes=plt.subplots(1,3,figsize=(20,6))
for ax,(col,t) in zip(axes,frames):
    pv=t.pivot(index='region',columns=col,values='units').fillna(0)
    pv=pv.div(pv.sum(axis=1),axis=0)*100
    pv.plot(kind='bar',stacked=True,ax=ax,legend=False);ax.set_title('Mix '+col);ax.set_ylabel('% unità')
    ax.tick_params(axis='x',rotation=35)
    ax.legend(title=col,fontsize=7,loc='upper left',bbox_to_anchor=(1.01,1))
fig.tight_layout();plt.show()
for col,t in frames:
    t=t.copy();t['share_within_region_pct']=100*t.units/t.groupby('region').units.transform('sum')
    data_table(t[['region',col,'units','share_within_region_pct']])